In [1]:
# Accomplishment so far:
# Retrieve chunks pertaining to a given policy name only

In [2]:
#Figure Out:
#1: What-if question contains category name - Easy
#2: What-if question contains more than one policy names - Tricky
#3: What to do for MRR when section is blank
#4: Issues when matched policy is None but relevant section exists
#5: Shall I restrategize my approach? Maybe it is better to add keyword in the test cases??

## Current MRR: 0.124

##Address both #5 with keywords in the test cases --> Update test cases
## Addressed #1 in the config

## New MRR: 0.5429

In [3]:
import os
import json
import glob
import math
import re

from dotenv import load_dotenv
from pathlib import Path
from typing import List
from openai import OpenAI
from pydantic import BaseModel, Field

In [4]:
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_experimental.text_splitter import SemanticChunker
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.messages import SystemMessage, HumanMessage

C:\Users\HP\AppData\Local\Temp\ipykernel_1124\2156210121.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, JSONLoader


### Vectorstore Creation

In [5]:
MODEL = "gpt-4.1-nano"
db_name = "vector_db"

In [6]:
RETRIEVAL_K = 20
CHUNK_SIZE = 1000

In [7]:
load_dotenv(override=True)

True

In [8]:
def normalize_section(text):
    # Remove extra spaces around hyphens and colons
    return re.sub(r'\s*[-:]\s*', lambda m: m.group().strip(), text).strip()

In [9]:
def load_json_with_root(filepath):
    with open(filepath, 'r') as f:
        full_data = json.load(f)
        policy_name = full_data.get("policy_name", "unknown")
        category = full_data.get("category", "unknown")
        source = full_data.get("source_path", "unknown").split('\\')[1]
        
    def metadata_func(record: dict, base_metadata: dict):
        base_metadata['policy_name'] = policy_name
        base_metadata['category'] = category
        base_metadata['source'] = source
        base_metadata['page_type'] = record.get("page_type", "unknown")
        return base_metadata
    
    return JSONLoader(
        file_path=filepath,
        jq_schema='.pages[]',
        content_key='text',
        metadata_func=metadata_func
    )

# Earlier jq schema: jq_schema='.pages[] | select(.page_type == "content")'

In [10]:
folders = glob.glob("knowledge-base/*")

documents = []
for folder in folders:
    loader = DirectoryLoader(folder, glob='**/*.json', loader_cls=load_json_with_root)
    folder_docs = loader.load()
    
    for doc in folder_docs:
        documents.append(doc)
        
print(len(documents))

539


#### Text Splitters

In [11]:
# documents[100]

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
len(chunks)

2414

In [13]:
# hf_embeddings = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [14]:
if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()
    
vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)

In [15]:
collection = vectorstore._collection
count = collection.count()
sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count} vectors with {dimensions:,} dimensions")

There are 2414 vectors with 1,536 dimensions


### Test Case Generation

In [19]:
class TestQuestion(BaseModel):
    test_id: str = Field(description="Unique identifier for the test case")
    query: str = Field(description="The user question to be asked to the RAG system")
    source_doc: str = Field(description="The policy document(s) where the answer should come from")
    expected_answer: str = Field(description="The correct ground truth answer for evaluation")
    keywords: list = Field(description="The sections in the document where the answer lives")
    question_type: str = Field(description="Category of the question")
    difficulty: str = Field(description="Complexity level of the question")

In [20]:
def load_tests() -> List[TestQuestion]:
    tests = []
    with open("tests.jsonl", 'r', encoding='utf-8') as f:
        for line in f:
            test = json.loads(line.strip())
            tests.append(TestQuestion(**test))
            
    return tests

In [21]:
tests = load_tests()
len(tests)

59

### LLM Answers

In [22]:
policies = []
categories = []
loc = "knowledge-base"
dirs = os.listdir(loc)

for d in dirs:
    categories.append(d.replace('_', ' ').lower())
    path = Path(f"{loc}/{d}")
    for file in path.iterdir():
        if file.is_file:
            policy = file.name.split('.')[0].lower()
            policies.append(policy.replace('_', ' '))
#             print(policy[:45], len(policy))

In [23]:
# retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [25]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the LIC (Life Insurance Corporation of India).
You are chatting with a user about LIC's insurance products only.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [26]:
# def answer_question(question: str, history):
#     docs = retriever.invoke(question)
#     context = "\n\n".join(doc.page_content for doc in docs)
#     system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
#     response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
#     return response.content

In [27]:
# answer_question("What are the two death benefit options available under LIC Digi Term?", [])

### Retrieval Evaluation

In [53]:
example = tests[58]
example

TestQuestion(test_id='TC_059', query='Which LIC plans fall under the term assurance category?', source_doc='multiple', expected_answer='The term assurance plans available in the knowledge base include Digi Term, New Jeevan Amar, Yuva Term, Bima Kavach, Digi Credit Life, Yuva Credit Life, and Saral Jeevan Bima. These are pure risk plans that provide death benefit protection without any maturity or savings component.', keywords=['term asurance'], question_type='cross_policy', difficulty='easy')

In [25]:
import re

In [30]:
# def normalize_section(text):
#     # Remove extra spaces around hyphens and colons
#     return re.sub(r'\s*[-:]\s*', lambda m: m.group().strip(), text).strip()

In [31]:
# # retriever = vectorstore.as_retriever()
# kwargs = {
#     "search_kwargs": {
#         "filter": {
#             "policy_name": next((p for p in policies if p in example.query.lower()), None)
#         }
#     }
# }
# documents = retriever.invoke(example.query, config=kwargs)

In [32]:
def calculate_mrr(docs, keyword):
    if not keyword:
        return None
    for rank, doc in enumerate(docs, start=1):
        keyword = normalize_section(keyword)
        if keyword.lower() in doc.page_content.lower(): #page_content is already normalized
            return 1.0 / rank
    return 0.0

In [33]:
# def calculate_dcg(docs, question, k):
#     dcg = 0
#     relevant_section = normalize_section(question.relevant_section)
#     relevences = [1 if relevant_section in doc.page_content else 0 for doc in docs]
    
#     for i in range(min(k, len(relevences))):
#         dcg += relevences[i] / math.log2(i+2)
#     return dcg

In [34]:
# def calculate_ndcg(docs, question, k):
#     relevant_section = normalize_section(question.relevant_section)
#     relevances = [1 if relevant_section in doc.page_content else 0 for doc in docs]
    
#     # DCG
#     dcg = sum(relevances[i] / math.log2(i + 2) for i in range(min(k, len(relevances))))
    
#     # Ideal DCG — best possible ranking
#     ideal_relevances = sorted(relevances, reverse=True)
#     idcg = sum(ideal_relevances[i] / math.log2(i + 2) for i in range(min(k, len(ideal_relevances))))
    
#     return dcg / idcg if idcg > 0 else 0.0

In [35]:
# def calculate_hit_rate(docs, question):
#     relevant_section = normalize_section(question.relevant_section)
#     tot_relevant_docs = sum(1 if relevant_section in doc.page_content else 0 for doc in docs)
#     return tot_relevant_docs / len(docs)

In [36]:
# def calculate_recall_k(docs, question, total_relevant, top_k=3):
#     relevant_section = normalize_section(question.relevant_section)
#     top_docs = sum(
#         1 for doc in docs[:top_k] 
#         if relevant_section in doc.page_content
#     )
#     return top_docs / total_relevant if total_relevant > 0 else 0.0

In [37]:
# calculate_mrr(documents, example)

In [38]:
# calculate_dcg(chunks, example, TOP_K)

In [39]:
# calculate_ndcg(documents, example, TOP_K)

In [40]:
# calculate_hit_rate(documents, example)

In [41]:
# calculate_recall_k(documents, example, chunks)

In [42]:
# Single test case
# mrr=0.25, ndcg=0.5, hitrate=40%

In [43]:
# Complexity => source_doc = "multiple", relevant_section = "multiple"

In [44]:
# kwargs = get_kwargs()
# retriever = vectorstore.as_retriever(search_kwargs={"k": RETRIEVAL_K})

In [45]:
def get_kwargs(question):
    p_matched = next((p for p in policies if p in question.lower()), None)
    c_matched = next((c for c in categories if c in question.lower()), None)
    if p_matched and c_matched:
        return {"filter": {"policy_name": p_matched, "category": c_matched}, "k": RETRIEVAL_K}
    elif p_matched:
        return {"filter": {"policy_name": p_matched}, "k": RETRIEVAL_K}
    elif c_matched:
        return {"filter": {"category": c_matched}, "k": RETRIEVAL_K}
    return {"k": RETRIEVAL_K}

In [46]:
def evaluate_test(test):
    kwargs = get_kwargs(test.query)
    retriever = vectorstore.as_retriever(search_kwargs=kwargs)
    docs = retriever.invoke(test.query)
    mrr = [calculate_mrr(docs, keyword) for keyword in test.keywords if keyword]
    avg_mrr = sum(mrr) / len(mrr) if mrr else 0.0
    return avg_mrr

In [47]:
def evaluate_all_tests(tests):
    for test in tests:
        result = evaluate_test(test)
        yield result

In [48]:
#evaluate_all_tests(tests)

In [49]:
tot_mrr = 0
count = 0
for result in evaluate_all_tests(tests):
    count += 1
    tot_mrr += result
    print(result, end=" ")
    
print()
print(tot_mrr / count)

0.625 0.75 0.6944444444444443 0.6923076923076922 0.125 0.16666666666666666 0.6666666666666666 0.0 1.0 0.75 1.0 0.5 1.0 0.5 0.5 0.5 0.41666666666666663 0.5555555555555556 0.08333333333333333 0.5384615384615384 0.625 1.0 0.0 0.5 0.0 1.0 0.12698412698412698 0.6 0.5 0.8333333333333334 1.0 0.0 0.625 0.6666666666666666 0.19607843137254902 0.3055555555555556 1.0 0.6 0.5 0.5 0.5 0.0 0.0 0.5 1.0 0.75 0.35 1.0 0.7333333333333334 0.1 0.5555555555555556 1.0 0.5357142857142857 0.5555555555555556 0.625 1.0 0.6 0.5833333333333334 0.0 
0.5429019108729978


In [56]:
# pip install langchain-experimental

In [50]:
example

TestQuestion(test_id='TC_001', query='What is the death benefit payable under LIC Jeevan Tarun?', source_doc='jeevan_tarun.json', expected_answer='The death benefit under Jeevan Tarun is the Sum Assured on Death along with vested Simple Reversionary Bonuses and Final Additional Bonus, if any. The Sum Assured on Death is defined as the higher of 125% of Basic Sum Assured or 7 times of Annualized Premium. This benefit shall not be less than 105% of the total premiums paid up to the date of death.', keywords=['death benefit', 'jeevan tarun'], question_type='factual', difficulty='easy')

In [52]:
matched = next((p for p in policies if p in example.query.lower()), None)
matched

['j', 'e', 'e', 'v', 'a', 'n', ' ', 't', 'a', 'r', 'u', 'n']

In [54]:
# Jeevan Tarun test case
p_matched = next((p for p in policies if p in example.query.lower()), None)
c_matched = next((c for c in categories if c in example.query.lower()), None)
print(f"Query: {example.query}")
print(f"Matched policy: {p_matched}")
print(f"Matched category: {c_matched}")
print(f"Section: {example.keywords}")

Query: Which LIC plans fall under the term assurance category?
Matched policy: None
Matched category: None
Section: ['term asurance']


In [64]:
kwargs = get_kwargs(example.query)
retriever = vectorstore.as_retriever(search_kwargs=kwargs)
docs = retriever.invoke(example.query)
print(f"Number of docs retrieved: {len(docs)}")
for doc in docs:
    print(doc.metadata)
    print()

Number of docs retrieved: 20
{'seq_num': 1, 'page_type': 'content', 'category': 'endowment plans', 'policy_name': 'bima lakshmi', 'source': 'lic_policies'}

{'page_type': 'content', 'source': 'lic_policies', 'category': 'endowment plans', 'policy_name': 'bima lakshmi', 'seq_num': 10}

{'policy_name': 'bima lakshmi', 'seq_num': 10, 'category': 'endowment plans', 'source': 'lic_policies', 'page_type': 'content'}

{'seq_num': 11, 'policy_name': 'bima lakshmi', 'page_type': 'content', 'source': 'lic_policies', 'category': 'endowment plans'}

{'page_type': 'content', 'source': 'lic_policies', 'policy_name': 'bima lakshmi', 'seq_num': 9, 'category': 'endowment plans'}

{'category': 'endowment plans', 'source': 'lic_policies', 'seq_num': 8, 'page_type': 'content', 'policy_name': 'bima lakshmi'}

{'page_type': 'content', 'category': 'endowment plans', 'seq_num': 6, 'policy_name': 'bima lakshmi', 'source': 'lic_policies'}

{'page_type': 'content', 'seq_num': 4, 'category': 'endowment plans', 'p

In [67]:
for i, doc in enumerate(docs):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content)
    print()

--- Chunk 1 ---
LIFE INSURANCE CORPORATION OF INDIA
(Established by the Life Insurance Corporation Act, 1956)
Registration Number:512
LIC's BIMA LAKSHMI (UIN:512N389V01)
(A Non-Par, Non-Linked, Life, Individual, Savings Plan)
PART-A
_____________________________
Space fo r Name a nd Addre ss of Pol icyholde r S pace for _A_d__d_re_s_s_ _a_n_d_ e_-_m__a_i_l _id_ _o_f_ B__r_a_n_c_h_ O ffice
Re:Your Policy No. _______________
We have pleasure in forwarding herewith the above Policy Document comprising of Part A to Part G along with Customer
Information Sheet (CIS), Benefit Illustration and Need Analysis documents.
We would also like to draw your kind attention to the information mentioned in the Schedule of the Policy and the benefits
available under the Policy.
Some of our plans have certain options (including Rider(s)) available under them. It is important that the options, if any, available

--- Chunk 2 ---
this Policy Document, whichever is applicable, shall be made.
2. On maturity:Ma

In [ ]:
policies

In [ ]:
from rapidfuzz import process, fuzz

In [ ]:
query = "brief me about female critical illness policy?"
best_match = process.extractOne(query, policies, scorer=fuzz.token_set_ratio)

In [ ]:
matches = process.extract(query, policies, scorer=fuzz.token_set_ratio)
matches[:2]